In [0]:
!pip install boto3 wfdb matplotlib 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.5.0
    Not uninstalling fsspec at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-92dfdeaf-aecf-4e8c-b777-7818617f505b
    Can't uninstall 'fsspec'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import scipy.signal as signal
import wfdb
import wfdb.processing 
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import gc, os, ast, shutil, tempfile, threading, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from rich import print
from rich.table import Table

# --- 0. CONFIGURACIÓN ESTRUCTURAL ---
BUCKET = 'physionet-open'
BUCKET_OUTPUT = 'shazam-proyecto-ecg'
S3_PREFIX = 'silver_12leads/clase_3_lbbb/'

TARGET_FS = 250
WINDOW_BACK = int(TARGET_FS * 0.3)
WINDOW_FORWARD = int(TARGET_FS * 0.5)
BEAT_LENGTH = WINDOW_BACK + WINDOW_FORWARD

# NUEVO: 12 Derivaciones estrictas
TARGET_LEADS = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

OUTPUT_DIR = '/Workspace/tmp/ecg_lbbb_expert'
OUTPUT_PDF = '/Workspace/tmp/ejemplos_lbbb_val.pdf'

MAX_FILES_CHAPMAN = 1500 
MAX_PATIENTS_PER_DATASET = 400 

all_beats = [] 
metadata_rows = []
_lock = threading.Lock()
MAX_WORKERS = 16 

thread_local = threading.local()

def get_s3_client():
    if not hasattr(thread_local, "s3_client"):
        config = Config(signature_version=UNSIGNED, max_pool_connections=10)
        thread_local.s3_client = boto3.client('s3', region_name='us-east-1', config=config)
    return thread_local.s3_client

# --- 1. LÓGICA DE VALIDACIÓN CLÍNICA (ESTRICTA + SCORE) ---
def get_multilead_qrs_duration(v1, v6, fs, center_idx):
    start = max(0, center_idx - int(0.12 * fs))
    end = min(len(v1), center_idx + int(0.18 * fs))
    
    w_v1, w_v6 = v1[start:end], v6[start:end]
    energy = np.abs(w_v1) + np.abs(w_v6)
    
    window_len = int(0.03 * fs)
    smoothed = np.convolve(energy, np.ones(window_len)/window_len, mode='same')
    
    threshold = np.max(smoothed) * 0.15 
    qrs_idx = np.where(smoothed > threshold)[0]
    
    if len(qrs_idx) < 2: return 0
    return qrs_idx[-1] - qrs_idx[0]

def evaluate_lbbb_score(beat, leads):
    try:
        idx_v1, idx_v6 = leads.index('V1'), leads.index('V6')
    except ValueError: 
        return 0, "missing_leads"

    v1, v6 = beat[idx_v1], beat[idx_v6]
    center = WINDOW_BACK
    score = 0
    
    v1_base = np.median(v1[:int(0.04 * TARGET_FS)])
    v6_base = np.median(v6[:int(0.04 * TARGET_FS)])
    v1_c, v6_c = v1 - v1_base, v6 - v6_base
    
    # CRITERIO 1: QRS Ancho (EXIGIMOS >= 120ms para score perfecto aquí)
    qrs_dur = get_multilead_qrs_duration(v1_c, v6_c, TARGET_FS, center)
    if qrs_dur >= int(0.120 * TARGET_FS): # Subido a 120ms (clínico)
        score += 1
        
    # CRITERIO 2: V1 predominantemente negativo
    if np.max(v1_c) < np.abs(np.min(v1_c)) * 1.5: 
        score += 1
        
    # CRITERIO 3: V6 predominantemente positivo
    if np.max(v6_c) > np.abs(np.min(v6_c)) * 1.0:
        score += 1
        
    start_qrs = max(0, center - int(0.05 * TARGET_FS))
    early_v6 = v6_c[start_qrs : center + int(0.02 * TARGET_FS)]
    if not (np.min(early_v6) < -0.3 and np.argmin(early_v6) < np.argmax(early_v6)):
        score += 1

    if score >= 3: confidence = "strong"
    elif score == 2: confidence = "weak"
    else: confidence = "reject"
        
    return score, confidence

# --- 2. PROCESAMIENTO DE SEÑALES ---
def process_ecg(raw_sig, fs):
    if fs != TARGET_FS:
        g = math.gcd(int(TARGET_FS), int(fs))
        raw_sig = signal.resample_poly(raw_sig, int(TARGET_FS) // g, int(fs) // g)
    
    nyq = 0.5 * TARGET_FS
    b, a = signal.butter(3, [0.5/nyq, 40.0/nyq], btype='band')
    filt = signal.filtfilt(b, a, raw_sig)
    return (filt - np.mean(filt)) / (np.std(filt) + 1e-8)

# --- 3. INGESTA DE REGISTROS (12 LEADS + SNAPPING + PEARSON) ---
def ingest_record(path, subject, patient_id, dataset, filter_label=None):
    s3_local = get_s3_client()
    tmp_dir = tempfile.mkdtemp()
    local_base = os.path.join(tmp_dir, "rec")
    try:
        s3_local.download_file(BUCKET, f"{path}.hea", f"{local_base}.hea")
        with open(f"{local_base}.hea", 'r') as f:
            dats = [l.split()[0] for l in f if '.' in l and not l.startswith('#')]
        
        p_dir = '/'.join(path.split('/')[:-1])
        for d in dats:
            s3_local.download_file(BUCKET, f"{p_dir}/{d}" if p_dir else d, os.path.join(tmp_dir, d))
        
        if dataset == "INCART" or filter_label:
            s3_local.download_file(BUCKET, f"{path}.atr", f"{local_base}.atr")

        rec = wfdb.rdrecord(local_base)
        names = [s.upper() for s in rec.sig_name]
        
        # MAPEADO ESTRICTO DE 12 DERIVACIONES
        mapping = {lead: None for lead in TARGET_LEADS}
        for t in mapping.keys():
            if t in names: mapping[t] = names.index(t)
            elif t == 'II' and 'MLII' in names: mapping[t] = names.index('MLII')
        
        if None in mapping.values(): return # Descartar si falta alguna

        idx = [mapping[lead] for lead in TARGET_LEADS]
        sigs = np.array([process_ecg(rec.p_signal[:, i], rec.fs) for i in idx])
        
        # EXTRACCIÓN DE PICOS BASE
        picos_raw = []
        if dataset == "INCART":
            try:
                ann = wfdb.rdann(local_base, 'atr')
                picos_temp = [ann.sample[i] for i in range(len(ann.symbol)) if ann.symbol[i] == 'L']
                if rec.fs != TARGET_FS:
                    picos_raw = [int(p * (TARGET_FS / rec.fs)) for p in picos_temp]
                else:
                    picos_raw = picos_temp
            except Exception: pass
        else:
            picos_raw = wfdb.processing.xqrs_detect(sigs[0], fs=TARGET_FS, verbose=False)

        # SEÑAL GLOBAL Y SNAPPING (Centrado RMS en 12 leads)
        rms_signal = np.sqrt(np.mean(sigs**2, axis=0))
        picos = []
        search_window = int(TARGET_FS * 0.05) # Búsqueda de ±50ms
        
        for p in picos_raw:
            start_p = max(0, p - search_window)
            end_p = min(sigs.shape[1], p + search_window)
            if start_p < end_p:
                # Ajuste exacto a la cresta del RMS
                local_max = np.argmax(rms_signal[start_p:end_p])
                picos.append(start_p + local_max)

        quality_map = {"PTB-XL": "strict", "INCART": "cardiologist", "CHAPMAN": "weak"}
        lbl_quality = quality_map.get(dataset, "unknown")
        
        # FASE 1: Extraer candidatos morfológicos
        candidates = []
        for p in picos:
            if p - WINDOW_BACK >= 0 and p + WINDOW_FORWARD < sigs.shape[1]:
                beat = sigs[:, p - WINDOW_BACK : p + WINDOW_FORWARD]
                # Evaluamos usando la lista dinámica de derivaciones
                score, conf = evaluate_lbbb_score(beat, TARGET_LEADS)
                
                if conf in ["strong", "weak"]:
                    final_conf = "strong" if dataset == "INCART" else conf
                    candidates.append({"beat": beat, "score": score, "conf": final_conf})

        # FASE 2: Filtro de Homogeneidad (Pearson) sobre V6 para aislar la morfología dominante
        extracted = []
        if len(candidates) >= 3:
            idx_v6 = TARGET_LEADS.index('V6') # Encontramos V6 dinámicamente
            v6_matrix = np.array([c["beat"][idx_v6] for c in candidates])
            template_v6 = np.median(v6_matrix, axis=0) 
            
            for c in candidates:
                corr = np.corrcoef(c["beat"][idx_v6], template_v6)[0, 1]
                if corr > 0.85: 
                    extracted.append((c["beat"], c["score"], c["conf"]))

        if extracted:
            with _lock:
                start_ptr = len(all_beats)
                beats_only = [e[0] for e in extracted]
                all_beats.extend(beats_only)
                
                metadata_rows.append({
                    'patient_id': patient_id, 
                    'dataset': dataset, 
                    'registro': subject,
                    'latidos': len(extracted), 
                    'start_idx': start_ptr,
                    'class_name': 'LBBB',
                    'label': 1,
                    'label_quality': lbl_quality,
                    'morph_score': np.mean([e[1] for e in extracted]),
                    'confidence': extracted[0][2] 
                })
            print(f"[green]✔ Procesado:[/green] {dataset} | ID: {patient_id} | Calidad: {lbl_quality} | [bold]{len(extracted)}[/bold] latidos 12-Leads.")
            
    except Exception as e: 
        print(f"[red]✖ Error:[/red] {dataset} | ID: {patient_id} | {str(e)}")
    finally: 
        shutil.rmtree(tmp_dir, ignore_errors=True)
# --- 4. ESCANEO Y EJECUCIÓN ---
print("[bold cyan]--- INICIANDO INGESTA MAESTRA LBBB (HIGH PERFORMANCE + PEARSON FILTER) ---[/bold cyan]")
tasks = []
s3_main = get_s3_client()

print("\n[bold yellow]--- INDEXANDO PTB-XL ---[/bold yellow]")
try:
    s3_main.download_file(BUCKET, 'ptb-xl/1.0.3/ptbxl_database.csv', 'ptbxl.csv')
    df_ptb = pd.read_csv('ptbxl.csv')
    df_ptb['scp_codes'] = df_ptb['scp_codes'].apply(ast.literal_eval)
    df_ptb_lbbb = df_ptb[df_ptb['scp_codes'].apply(lambda x: ('LBBB' in x or 'CLBBB' in x or 'ILBBB' in x) and 'PACE' not in x)]
    
    for _, r in df_ptb_lbbb.iterrows():
        tasks.append((f"ptb-xl/1.0.3/{r['filename_hr']}", r['filename_hr'], f"PTB_{r['patient_id']}", "PTB-XL", False))
    print(f"[bold green]🎯 Total PTB-XL encolados: {len(df_ptb_lbbb)}[/bold green]")
except Exception as e: 
    print(f"[bold red]Error fatal en PTB-XL: {e}[/bold red]")

print("\n[bold yellow]--- INDEXANDO INCART ---[/bold yellow]")
for i in range(1, 76):
    file_name = f"I{i:02d}"
    tasks.append((f"incartdb/1.0.0/{file_name}", file_name, f"INCART_P{i}", "INCART", True))
print(f"[bold green]🎯 Total INCART encolados: 75[/bold green]")

print("\n[bold yellow]--- INDEXANDO CHAPMAN (Limitado) ---[/bold yellow]")
pag = s3_main.get_paginator('list_objects_v2')
all_chap_heas = []
for page in pag.paginate(Bucket=BUCKET, Prefix='ecg-arrhythmia/1.0.0/WFDBRecords/'):
    all_chap_heas.extend([obj['Key'] for obj in page.get('Contents', []) if obj['Key'].endswith('.hea')])

def check_chap_header(key):
    try:
        s3_temp = get_s3_client()
        res = s3_temp.get_object(Bucket=BUCKET, Key=key, Range='bytes=0-2048')
        if b'164909002' in res['Body'].read():
            return key
    except Exception: pass
    return None

chap_found = 0
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = [executor.submit(check_chap_header, k) for k in all_chap_heas]
    for future in as_completed(futures):
        result_key = future.result()
        if result_key:
            base = result_key.replace('.hea','')
            file_name = base.split('/')[-1]
            tasks.append((base, file_name, f"CHAP_{file_name}", "CHAPMAN", False))
            chap_found += 1
            if chap_found >= MAX_FILES_CHAPMAN:
                print(f"[yellow]Chapman truncado a {MAX_FILES_CHAPMAN} archivos para evitar sobre-representación.[/yellow]")
                for f in futures: f.cancel()
                break

print(f"\n[bold magenta]🚀 INICIANDO DESCARGA Y EXTRACCIÓN ({len(tasks)} TAREAS | {MAX_WORKERS} WORKERS)...[/bold magenta]")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as e:
    list(as_completed([e.submit(ingest_record, *t) for t in tasks]))

# --- 5. SPLIT Y BALANCEO ---
if all_beats:
    print("\n[bold cyan]--- PROCESAMIENTO FINALIZADO. APLICANDO BALANCEO CLÍNICO ---[/bold cyan]")
    df_meta = pd.DataFrame(metadata_rows)
    
    # 5.1 BALANCEO ANTI-DOMINANCIA
    balanced_patients = []
    for ds in df_meta['dataset'].unique():
        ds_pats = df_meta[df_meta['dataset'] == ds]['patient_id'].unique()
        if len(ds_pats) > MAX_PATIENTS_PER_DATASET:
            print(f"[yellow]Reduciendo {ds} de {len(ds_pats)} a {MAX_PATIENTS_PER_DATASET} pacientes...[/yellow]")
            ds_pats = np.random.choice(ds_pats, MAX_PATIENTS_PER_DATASET, replace=False)
        balanced_patients.extend(ds_pats)
        
    df_meta = df_meta[df_meta['patient_id'].isin(balanced_patients)].copy()
    
    # 5.2 SPLIT
    pats = df_meta['patient_id'].unique()
    if len(pats) >= 3:
        train_p, test_p = train_test_split(pats, test_size=0.2, random_state=42)
        if len(test_p) > 1:
            val_p, test_p = train_test_split(test_p, test_size=0.5, random_state=42)
        else:
            val_p = [] 
    else:
        train_p, val_p, test_p = pats, [], []

    def set_split(p):
        if p in train_p: return 'train'
        if p in val_p: return 'val'
        return 'test'
        
    df_meta['split'] = df_meta['patient_id'].apply(set_split)

    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for s in ['train', 'val', 'test']:
        idx_list, labels, patients, datasets, qualities, confidences = [], [], [], [], [], []
        subset = df_meta[df_meta['split'] == s]
        
        if subset.empty: continue
            
        for _, r in subset.iterrows():
            beat_range = range(r['start_idx'], r['start_idx'] + r['latidos'])
            idx_list.extend(beat_range)
            labels.extend([r['label']] * r['latidos'])
            patients.extend([r['patient_id']] * r['latidos'])
            datasets.extend([r['dataset']] * r['latidos'])
            qualities.extend([r['label_quality']] * r['latidos'])
            confidences.extend([r['confidence']] * r['latidos'])
            
        if idx_list:
            X = np.array([all_beats[i] for i in idx_list], dtype=np.float32)
            y = np.array(labels, dtype=np.int32)
            
            split_dir = os.path.join(OUTPUT_DIR, s)
            os.makedirs(split_dir, exist_ok=True)
            np.savez_compressed(f"{split_dir}/lbbb_signals.npz", x=X, y=y)
            
            beat_meta = pd.DataFrame({
                'patient_id': patients,
                'dataset': datasets,
                'class_name': 'LBBB',
                'label': 1,
                'label_quality': qualities,
                'morph_confidence': confidences
            })
            beat_meta.to_csv(f"{split_dir}/lbbb_meta.csv", index=False)
            print(f"[bold green]💾 Exportado {s}: {X.shape[0]} latidos en '{s}/'[/bold green]")

    table = Table(title="Resumen Final (Post-Balanceo)")
    table.add_column("Dataset", justify="left", style="cyan")
    table.add_column("Total Beats", justify="right", style="green")

    for ds in df_meta['dataset'].unique():
        beats_total = df_meta[df_meta['dataset'] == ds]['latidos'].sum()
        table.add_row(ds, str(beats_total))
    print(table)

   # --- NUEVA GENERACIÓN DE PDF: 12 DERIVACIONES LBBB + GRILLA ECG ---
    print(f"\n[yellow]Generando PDF de validación visual (12 Derivaciones LBBB): {OUTPUT_PDF}[/yellow]")
    t = np.linspace(-WINDOW_BACK / TARGET_FS, WINDOW_FORWARD / TARGET_FS, BEAT_LENGTH, endpoint=False)
    
    with PdfPages(OUTPUT_PDF) as pdf:
        for ds in df_meta['dataset'].unique():
            # Seleccionamos 50 pacientes top por base de datos para auditar
            samples = df_meta[df_meta['dataset'] == ds].head(50)
            if samples.empty: continue
            
            for r_idx, (_, row) in enumerate(samples.iterrows()):
                beat = all_beats[row['start_idx']]
                pat_id = row['patient_id']
                conf = row['confidence']

                # Grilla de 4 filas x 3 columnas (Página completa por latido)
                fig, axs = plt.subplots(4, 3, figsize=(18, 16), sharex=True)
                fig.suptitle(f"Dataset: {ds} - Auditoría LBBB | 12 Derivaciones\nPaciente: {pat_id} | Confianza: {conf.upper()}", 
                             fontsize=18, fontweight='bold', y=0.96)

                axs = axs.flatten()
                for c_idx in range(12):
                    ax = axs[c_idx]
                    
                    # Pintamos en azul clínico
                    ax.plot(t, beat[c_idx], color='#1f77b4', linewidth=1.5)
                    
                    # Grilla Milimétrica ECG
                    ax.grid(True, which='both', color='red', linestyle='--', linewidth=0.5, alpha=0.3)
                    ax.minorticks_on()
                    ax.grid(True, which='minor', color='red', linestyle=':', linewidth=0.2, alpha=0.2)
                    
                    # Marcadores: Centro R y Zona LBBB mínima (120ms)
                    ax.axvline(0, color='black', linewidth=1.2, linestyle='-')
                    ax.axvspan(0.0, 0.12, color='gray', alpha=0.15, label='Min QRS (120ms)')
                    
                    ax.set_title(f"Derivación {TARGET_LEADS[c_idx]}", fontsize=12, fontweight='bold')
                    ax.set_xlim([-0.2, 0.4]) # Rango enfocado en el QRS/ST

                plt.tight_layout(rect=[0, 0.03, 1, 0.93])
                pdf.savefig(fig)
                plt.close()

    # --- TU GUARDADO EXACTO EN DATABRICKS ---
    try:
        s3_dest = f"s3://{BUCKET_OUTPUT}/{S3_PREFIX}"
        print(f"\n[cyan]Subiendo a {s3_dest}...[/cyan]")
        for split_folder in ['train', 'val', 'test']:
            local_split_dir = os.path.join(OUTPUT_DIR, split_folder)
            if os.path.exists(local_split_dir):
                s3_split_dest = f"{s3_dest}{split_folder}/" 
                for f in os.listdir(local_split_dir):
                    dbutils.fs.cp(f"file:{os.path.join(local_split_dir, f)}", f"{s3_split_dest}{f}")
                    
        if os.path.exists(OUTPUT_PDF):
            dbutils.fs.cp(f"file:{OUTPUT_PDF}", f"{s3_dest}{os.path.basename(OUTPUT_PDF)}")
            print(f"[bold green]✅ PDF subido a {s3_dest}{os.path.basename(OUTPUT_PDF)}[/bold green]")
            
    except NameError:
        print("[yellow]dbutils no está definido. Saltando subida a S3 (ejecución local detectada).[/yellow]")

else:
    print("[bold red]ERROR: No se encontraron latidos.[/bold red]")

gc.collect()

--- INICIANDO INGESTA MAESTRA LBBB (HIGH PERFORMANCE + PEARSON FILTER) ---

--- INDEXANDO PTB-XL ---

🎯 Total PTB-XL encolados: 612

--- INDEXANDO INCART ---

🎯 Total INCART encolados: 75

...

      Resumen Final      
     (Post-Balanceo)     
┏━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Dataset ┃ Total Beats ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━┩
│ PTB-XL  │        5189 │
│ CHAPMAN │        2467 │
└─────────┴─────────────┘

Generando PDF de validación visual (12 Derivaciones LBBB): /Workspace/tmp/ejemplos_lbbb_val.pdf

Subiendo a s3://shazam-proyecto-ecg/silver_12leads/clase_3_lbbb/...

✅ PDF subido a s3://shazam-proyecto-ecg/silver_12leads/clase_3_lbbb/ejemplos_lbbb_val.pdf

1661